In [2]:
import metapredict as meta
import pandas as pd

df = pd.read_csv('../outputs/miniexon_aa_seq_marked_curated.csv')

In [3]:
result_dict = {}
disorder_score_dict = {}
for event, sequence in zip(df['EVENT'], df['aa_seq_marked']):

    seq = ''.join(sequence.split('#'))
    start = len(sequence.split('#')[0]) 
    end = start+len(sequence.split('#')[1]) 
    x = meta.predict_disorder(seq, return_domains=True)
    result_dict[event] = x
    disorder_score_dict[event] = (start,end, x.disordered_domain_boundaries, x.disorder[start:end].mean(),x.disorder.mean())
    

In [4]:
df_disorder = pd.DataFrame(disorder_score_dict, index = ['exon_start', 'exon_end', 'disordered_domain_boundaries', 'exon_disorder_score', 'protein_disorder_score']).T

In [5]:
df = df.set_index('EVENT').join(df_disorder)

In [6]:
in_disorder = {}
disorder_seq = {}
for i in range(len(df)):
    row = df.iloc[i]
    f = 0
    for d in row['disordered_domain_boundaries']:
        if (d[0]<=row['exon_start']<=d[1]) or (d[0]<=row['exon_end']<=d[1]):
            f = True
            mut = list(row['aa_seq']) 
            for i in range(row['exon_start'], row['exon_end']):
                mut[i] = ' '
            mut = ''.join(mut)
            mut_disorder = mut[d[0]:d[1]].replace(' ','')
            wt_disordr = row['aa_seq'][d[0]:d[1]]
            disorder_seq[row.name] = (wt_disordr, mut_disorder)
    in_disorder[row.name] = f

In [7]:
df = df.join(pd.DataFrame(disorder_seq, index = ['wt_disordr', 'mut_disorder']).T).join(pd.DataFrame(in_disorder, index = ['in_disorder']).T)


In [8]:
df_in_disorder = df[df['in_disorder'] == 1]

In [12]:
import random
import numpy as np
import json
random.seed = 41
random_disorder = {}
for event in df_disorder.index:
    result = result_dict[event].disorder
    s = df.loc[event]
    n = len(s['exon_aa_seq'].split('*')[0])
    m = len(s['aa_seq']) - n
    random_numbers = [random.randint(0, m) for _ in range(100)]
    random_disorder[event] = np.mean([result[i:i+n].mean() for i in random_numbers])
    
    

In [26]:
df['IDR #'] = df.apply(lambda x: len(x['disordered_domain_boundaries']), axis=1)

/tmp/ipykernel_3040467/1918793454.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['IDR #'] = df.apply(lambda x: len(x['disordered_domain_boundaries']), axis=1)


In [28]:
df.to_csv('../outputs/miniexon_aa_seq_marked_curated_IDR.csv')